# 01 - ARIMA Models via State-Space Representation

## Introduction

Any ARIMA(p,d,q) model can be written as a **linear Gaussian state-space model** (SSM).
This is powerful because the Kalman filter provides:

1. **Exact maximum likelihood** estimation via the prediction error decomposition
2. **Missing data handling** — the filter naturally skips missing observations
3. **State extraction** — filtered and smoothed estimates of latent components
4. **Forecasting** with proper uncertainty quantification

### General ARIMA(p,d,q) in State-Space Form

After differencing $d$ times, the ARMA(p,q) process $w_t = \Delta^d y_t$ has the companion form:

$$
\begin{aligned}
\boldsymbol{\alpha}_{t+1} &= \mathbf{T} \boldsymbol{\alpha}_t + \mathbf{R} \eta_t, \quad \eta_t \sim N(0, \sigma^2) \\
w_t &= \mathbf{Z} \boldsymbol{\alpha}_t
\end{aligned}
$$

where the state dimension is $m = \max(p, q+1)$ and:

| Matrix | Definition |
|--------|-----------|
| $\mathbf{T}$ | Companion form with AR coefficients $\phi_1, \ldots, \phi_p$ in first row |
| $\mathbf{Z}$ | $[1, \theta_1, \theta_2, \ldots, \theta_{q}, 0, \ldots]$ (MA coefficients) |
| $\mathbf{R}$ | $[1, 0, \ldots, 0]'$ (selection vector) |
| $\mathbf{Q}$ | $[\sigma^2]$ (innovation variance) |
| $\mathbf{H}$ | $[0]$ (no additional observation noise) |

This notebook demonstrates four models and missing data handling using **kalmanbox**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

from kalmanbox import ARIMA_SSM, LocalLevel
from kalmanbox.datasets import load_dataset
from kalmanbox.filters.kalman import KalmanFilter
from kalmanbox.core.representation import StateSpaceRepresentation

import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA as SM_ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
})
print('Imports OK')

## Example 1: ARIMA(0,1,1) = Local Level Model (Nile Data)

The ARIMA(0,1,1) model is:

$$y_t = y_{t-1} + \varepsilon_t + \theta \varepsilon_{t-1}, \quad \varepsilon_t \sim N(0, \sigma^2)$$

After differencing: $w_t = \Delta y_t = \varepsilon_t + \theta \varepsilon_{t-1}$, which is MA(1).

This is **exactly equivalent** to the **local level model**:

$$
\begin{aligned}
y_t &= \mu_t + \varepsilon_t^*, \quad \varepsilon_t^* \sim N(0, \sigma^2_\varepsilon) \\
\mu_t &= \mu_{t-1} + \eta_t, \quad \eta_t \sim N(0, \sigma^2_\eta)
\end{aligned}
$$

The mapping between parameters is:
- $\sigma^2 = \sigma^2_\varepsilon + \sigma^2_\eta$
- $\theta = -\sigma^2_\varepsilon / \sigma^2$ (negative of signal-to-noise related)

### State-Space Matrices for ARIMA(0,1,1)

After differencing ($m = \max(0, 1+1) = 2$ but kalmanbox uses $m = \max(0, 1) = 1$; for MA(1) the state dim is 1 with $\mathbf{Z} = [1, \theta]$):

| Matrix | Value |
|--------|-------|
| $\mathbf{T}$ | $\begin{bmatrix} 0 \\ 1 & 0 \end{bmatrix}$ (companion form, no AR) |
| $\mathbf{Z}$ | $[1, \theta]$ |
| $\mathbf{R}$ | $[1, 0]'$ |
| $\mathbf{Q}$ | $[\sigma^2]$ |
| $\mathbf{H}$ | $[0]$ |

In [ ]:
# Load Nile data
df_nile = load_dataset('nile')
y_nile = df_nile['volume'].to_numpy(dtype=np.float64)
years = df_nile['year'].to_numpy()
print(f'Nile data: {len(y_nile)} observations, {years[0]}-{years[-1]}')

# --- Fit ARIMA(0,1,1) via kalmanbox ---
arima_011 = ARIMA_SSM(y_nile, order=(0, 1, 1))
res_arima = arima_011.fit()

print('\n=== ARIMA(0,1,1) via kalmanbox (state-space) ===')
print(res_arima.summary())

# --- Fit Local Level model via kalmanbox ---
ll_model = LocalLevel(y_nile)
res_ll = ll_model.fit()

print('\n=== Local Level Model via kalmanbox ===')
print(res_ll.summary())

In [ ]:
# Show the equivalence between ARIMA(0,1,1) and Local Level parameters
theta = res_arima.params[0]  # theta_1 (MA coefficient)
sigma2 = res_arima.params[1]  # sigma2 (innovation variance)

# Mapping: ARIMA(0,1,1) -> Local Level
# The reduced form of the local level model is ARIMA(0,1,1) with:
#   sigma2_total = sigma2_obs + sigma2_level
#   theta satisfies: theta * sigma2_total = -sigma2_obs  (approximately)
# More precisely, for MA(1) of differenced local level:
#   gamma(0) = sigma2_obs + 2*sigma2_level  (variance of diff series)
#   gamma(1) = -sigma2_level                (autocovariance at lag 1)
#   theta satisfies: gamma(1)/gamma(0) = theta/(1+theta^2)

# From ARIMA parameters, derive local level equivalents
# MA(1) model: w_t = e_t + theta*e_{t-1}, Var(w_t) = sigma2*(1+theta^2)
# gamma(0) = sigma2 * (1 + theta**2)
# gamma(1) = sigma2 * theta
# For local level: gamma(0) = sigma2_obs + 2*sigma2_level, gamma(1) = -sigma2_level
# So: sigma2_level = -sigma2 * theta
#     sigma2_obs = sigma2 * (1 + theta**2) - 2*sigma2_level = sigma2*(1+theta^2+2*theta) = sigma2*(1+theta)^2

sigma2_level_equiv = -sigma2 * theta
sigma2_obs_equiv = sigma2 * (1 + theta)**2

print('=== Parameter Equivalence: ARIMA(0,1,1) <-> Local Level ===\n')
print(f'ARIMA(0,1,1) parameters:')
print(f'  theta_1 = {theta:.4f}')
print(f'  sigma2  = {sigma2:.2f}')
print(f'\nDerived Local Level parameters (from ARIMA):')
print(f'  sigma2_obs   = sigma2*(1+theta)^2 = {sigma2_obs_equiv:.2f}')
print(f'  sigma2_level = -sigma2*theta       = {sigma2_level_equiv:.2f}')
print(f'\nDirect Local Level estimates:')
print(f'  sigma2_obs   = {res_ll.params[0]:.2f}')
print(f'  sigma2_level = {res_ll.params[1]:.2f}')
print(f'\nLog-likelihood comparison:')
print(f'  ARIMA(0,1,1): {res_arima.loglike:.4f} (on differenced series, n={len(y_nile)-1})')
print(f'  Local Level:  {res_ll.loglike:.4f} (on original series, n={len(y_nile)})')

# Compare with statsmodels
sm_arima = SM_ARIMA(y_nile, order=(0, 1, 1))
sm_res = sm_arima.fit()
print(f'\nstatsmodels ARIMA(0,1,1):')
print(f'  theta_1 = {sm_res.params[0]:.4f}')
print(f'  sigma2  = {sm_res.params[1]:.2f}')

In [ ]:
# Visualize: ARIMA(0,1,1) filtered state on differenced series
w_nile = np.diff(y_nile)  # differenced series
filtered_arima = res_arima.filtered_state[:, 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Differenced series + ARIMA filtered state
axes[0].plot(years[1:], w_nile, 'k.', markersize=4, alpha=0.5, label='$\\Delta y_t$ (observed)')
axes[0].plot(years[1:], filtered_arima, 'b-', linewidth=1.5, label='ARIMA(0,1,1) filtered')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('$\\Delta$ Flow')
axes[0].set_title('ARIMA(0,1,1) on Differenced Nile Data')
axes[0].legend()
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)

# Right: Local Level smoothed state on original series
smoothed_ll = res_ll.smoothed_state[:, 0]
axes[1].plot(years, y_nile, 'k.', markersize=4, alpha=0.5, label='Observed')
axes[1].plot(years, smoothed_ll, 'r-', linewidth=1.5, label='Local Level (smoothed)')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Annual Flow')
axes[1].set_title('Equivalent Local Level Model on Nile Data')
axes[1].legend()

plt.suptitle('ARIMA(0,1,1) $\\equiv$ Local Level Model', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Example 2: AR(1) via State-Space

The AR(1) model $y_t = \phi y_t + \varepsilon_t$ has the simplest possible state-space form:

$$
\begin{aligned}
\alpha_{t+1} &= \phi \, \alpha_t + \eta_t, \quad \eta_t \sim N(0, \sigma^2) \\
y_t &= \alpha_t
\end{aligned}
$$

### State-Space Matrices for AR(1) = ARIMA(1,0,0)

| Matrix | Value | Dimension |
|--------|-------|-----------|
| $\mathbf{T}$ | $[\phi]$ | $1 \times 1$ |
| $\mathbf{Z}$ | $[1]$ | $1 \times 1$ |
| $\mathbf{R}$ | $[1]$ | $1 \times 1$ |
| $\mathbf{Q}$ | $[\sigma^2]$ | $1 \times 1$ |
| $\mathbf{H}$ | $[0]$ | $1 \times 1$ |

The state is simply the process itself: $\alpha_t = y_t$.

In [ ]:
# Use Nile data (demeaned) for AR(1) demonstration
y_ar = y_nile - y_nile.mean()

# --- Fit AR(1) via kalmanbox ARIMA_SSM ---
ar1_model = ARIMA_SSM(y_ar, order=(1, 0, 0))
res_ar1 = ar1_model.fit()

print('=== AR(1) via kalmanbox ARIMA_SSM ===')
print(res_ar1.summary())

# Show the SSM matrices explicitly
ssm_ar1 = ar1_model._build_ssm(res_ar1.params)
print(f'\nState-Space Matrices:')
print(f'  T = {ssm_ar1.T}  (AR coefficient phi)')
print(f'  Z = {ssm_ar1.Z}  (observation = state)')
print(f'  R = {ssm_ar1.R.flatten()}  (selection)')
print(f'  Q = {ssm_ar1.Q}  (innovation variance)')
print(f'  H = {ssm_ar1.H}  (no obs noise)')

# --- Compare with statsmodels ---
sm_ar1 = SM_ARIMA(y_ar, order=(1, 0, 0), trend='n')
sm_res_ar1 = sm_ar1.fit()

print(f'\n=== Comparison: kalmanbox vs statsmodels ===')
print(f'{"Parameter":<15} {"kalmanbox":>12} {"statsmodels":>12}')
print(f'{"-"*40}')
print(f'{"phi_1":<15} {res_ar1.params[0]:>12.4f} {sm_res_ar1.params[0]:>12.4f}')
print(f'{"sigma2":<15} {res_ar1.params[1]:>12.2f} {sm_res_ar1.params[1]:>12.2f}')
print(f'{"Log-lik":<15} {res_ar1.loglike:>12.4f} {sm_res_ar1.llf:>12.4f}')

In [ ]:
# Visualize AR(1) filtered state
filtered_ar1 = res_ar1.filtered_state[:, 0]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(years, y_ar, 'k.', markersize=4, alpha=0.5, label='Observed (demeaned)')
ax.plot(years, filtered_ar1, 'b-', linewidth=1.5, label='AR(1) filtered state')
ax.fill_between(
    years,
    filtered_ar1 - 1.96 * np.sqrt(res_ar1.filtered_cov[:, 0, 0]),
    filtered_ar1 + 1.96 * np.sqrt(res_ar1.filtered_cov[:, 0, 0]),
    alpha=0.2, color='blue', label='95% CI'
)
ax.set_xlabel('Year')
ax.set_ylabel('Demeaned Flow')
ax.set_title(f'AR(1) via State-Space: $\\hat{{\\phi}}$ = {res_ar1.params[0]:.3f}')
ax.legend()
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Example 3: ARIMA(1,1,1) via State-Space

The ARIMA(1,1,1) model combines AR and MA components with differencing:

$$\Delta y_t = \phi \, \Delta y_{t-1} + \varepsilon_t + \theta \, \varepsilon_{t-1}$$

After differencing, this is an ARMA(1,1) process. The state dimension is
$m = \max(1, 2) = 2$ and the companion form is:

$$
\mathbf{T} = \begin{bmatrix} \phi & 0 \\ 1 & 0 \end{bmatrix}, \quad
\mathbf{Z} = [1, \theta], \quad
\mathbf{R} = \begin{bmatrix} 1 \\ 0 \end{bmatrix}
$$

In [ ]:
# --- Fit ARIMA(1,1,1) via kalmanbox ---
arima_111 = ARIMA_SSM(y_nile, order=(1, 1, 1))
res_111 = arima_111.fit()

print('=== ARIMA(1,1,1) via kalmanbox ===')
print(res_111.summary())

# Show explicit SSM matrices
ssm_111 = arima_111._build_ssm(res_111.params)
print(f'\nState-Space Matrices (m={ssm_111.k_states}):')
print(f'  T =\n{ssm_111.T}')
print(f'  Z = {ssm_111.Z}')
print(f'  R = {ssm_111.R.flatten()}')
print(f'  Q = {ssm_111.Q}')

# --- Compare with statsmodels ---
sm_111 = SM_ARIMA(y_nile, order=(1, 1, 1))
sm_res_111 = sm_111.fit()

print(f'\n=== Comparison: kalmanbox vs statsmodels ===')
print(f'{"Parameter":<15} {"kalmanbox":>12} {"statsmodels":>12}')
print(f'{"-"*40}')
print(f'{"phi_1":<15} {res_111.params[0]:>12.4f} {sm_res_111.params[0]:>12.4f}')
print(f'{"theta_1":<15} {res_111.params[1]:>12.4f} {sm_res_111.params[1]:>12.4f}')
print(f'{"sigma2":<15} {res_111.params[2]:>12.2f} {sm_res_111.params[2]:>12.2f}')
print(f'{"Log-lik":<15} {res_111.loglike:>12.4f} {sm_res_111.llf:>12.4f}')

## Example 4: Airline Model — ARIMA(0,1,1)(0,1,1)$_{12}$

The classic **airline model** (Box & Jenkins, 1976) for the international airline passengers data:

$$(1 - B)(1 - B^{12}) y_t = (1 + \theta B)(1 + \Theta B^{12}) \varepsilon_t$$

After both regular and seasonal differencing, the resulting process is a multiplicative
MA model. The state-space representation uses the companion form with total MA order
$q_{total} = 1 + 12 = 13$ (non-seasonal MA at lag 1, seasonal MA at lag 12, and their
interaction potentially at lag 13).

This is a powerful demonstration of the SSM approach — seasonal ARIMA models that are
complex to write in reduced form become straightforward companion-form state-space models.

In [ ]:
# Load airline data
df_air = load_dataset('airline')
print(f'Airline data: {df_air.shape[0]} observations')
print(f'Columns: {df_air.columns.tolist()}')
print(df_air.head())

# Use log-transformed passengers (classic Box-Jenkins approach)
y_air = np.log(df_air['passengers'].to_numpy(dtype=np.float64))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(df_air['passengers'], 'b-', linewidth=0.8)
axes[0].set_title('Airline Passengers (Original)')
axes[0].set_ylabel('Passengers')

axes[1].plot(y_air, 'b-', linewidth=0.8)
axes[1].set_title('Log Airline Passengers')
axes[1].set_ylabel('Log Passengers')

plt.tight_layout()
plt.show()

In [ ]:
# --- Fit ARIMA(0,1,1)(0,1,1)_12 via kalmanbox ---
airline_model = ARIMA_SSM(y_air, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12))
res_airline = airline_model.fit()

print('=== Airline Model ARIMA(0,1,1)(0,1,1)_12 via kalmanbox ===')
print(res_airline.summary())

# Show SSM matrices structure
ssm_air = airline_model._build_ssm(res_airline.params)
print(f'\nState dimension: m = {ssm_air.k_states}')
print(f'T shape: {ssm_air.T.shape}')
print(f'Z shape: {ssm_air.Z.shape}')
print(f'Z (non-zero entries): {[(i, ssm_air.Z[0, i]) for i in range(ssm_air.k_states) if abs(ssm_air.Z[0, i]) > 1e-10]}')

# --- Compare with statsmodels SARIMAX ---
sm_airline = SARIMAX(y_air, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12))
sm_res_air = sm_airline.fit(disp=False)

print(f'\n=== Comparison: kalmanbox vs statsmodels SARIMAX ===')
kb_names = res_airline.param_names
sm_params = sm_res_air.params

print(f'{"Parameter":<15} {"kalmanbox":>12} {"statsmodels":>12}')
print(f'{"-"*40}')
# kalmanbox: theta_1, theta_12, sigma2
# statsmodels: ma.L1, ma.S.L12, sigma2
print(f'{"theta_1 (MA1)":<15} {res_airline.params[0]:>12.4f} {sm_params[0]:>12.4f}')
print(f'{"theta_12 (SMA)":<15} {res_airline.params[1]:>12.4f} {sm_params[1]:>12.4f}')
print(f'{"sigma2":<15} {res_airline.params[2]:>12.6f} {sm_params[2]:>12.6f}')
print(f'{"Log-lik":<15} {res_airline.loglike:>12.4f} {sm_res_air.llf:>12.4f}')

In [ ]:
# Visualize airline model: fitted values on differenced series
w_air = airline_model._differenced_endog
filtered_air = res_airline.filtered_state[:, 0]

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Differenced series + filtered
axes[0].plot(w_air, 'k-', linewidth=0.6, alpha=0.6, label='Differenced (observed)')
axes[0].plot(filtered_air, 'b-', linewidth=1.2, label='Filtered state')
axes[0].set_title('Airline Model: Filtered State on Differenced Series')
axes[0].set_ylabel('$\\Delta \\Delta_{12} \\log(y_t)$')
axes[0].legend()

# Residuals
resid_air = res_airline.residuals[:, 0]
axes[1].plot(resid_air, 'k-', linewidth=0.6)
axes[1].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1].set_title('Prediction Errors (Residuals)')
axes[1].set_ylabel('$v_t$')
axes[1].set_xlabel('Observation')

plt.tight_layout()
plt.show()

## Missing Data Handling via Kalman Filter

One of the key advantages of the state-space/Kalman filter approach is **natural
handling of missing data**. When $y_t$ is missing (NaN), the Kalman filter simply
skips the update step:

- **Prediction**: $\hat{\alpha}_{t|t-1} = T \hat{\alpha}_{t-1|t-1} + c$ (unchanged)
- **No update**: $\hat{\alpha}_{t|t} = \hat{\alpha}_{t|t-1}$ (filtered = predicted)
- The covariance grows: $P_{t|t} = P_{t|t-1}$ (more uncertainty)

This means we can estimate parameters and extract states even with gaps in the data,
without any imputation or special treatment. Classical ARIMA estimation (e.g., CSS)
cannot handle this directly.

We demonstrate by randomly removing 10% of the Nile data and refitting the ARIMA(0,1,1).

In [ ]:
# Create version with 10% missing data
rng = np.random.default_rng(42)
y_missing = y_nile.copy()
n_missing = int(0.10 * len(y_nile))
missing_idx = rng.choice(len(y_nile), size=n_missing, replace=False)
y_missing[missing_idx] = np.nan

print(f'Total observations: {len(y_nile)}')
print(f'Missing: {n_missing} ({100*n_missing/len(y_nile):.0f}%)')
print(f'Missing indices: {sorted(missing_idx)}')

# Fit ARIMA(0,1,1) on data with missing values
# Note: ARIMA_SSM differences the data, so we need to handle NaN propagation.
# Instead, we use the Local Level model which operates on the original series.
ll_missing = LocalLevel(y_missing)
res_missing = ll_missing.fit()

# Fit on complete data for comparison
ll_complete = LocalLevel(y_nile)
res_complete = ll_complete.fit()

print(f'\n=== Parameter Comparison: Complete vs Missing Data ===')
print(f'{"Parameter":<18} {"Complete":>12} {"10% Missing":>12} {"Diff (%)":>10}')
print(f'{"-"*55}')
for i, name in enumerate(res_complete.param_names):
    pct = 100 * (res_missing.params[i] - res_complete.params[i]) / res_complete.params[i]
    print(f'{name:<18} {res_complete.params[i]:>12.2f} {res_missing.params[i]:>12.2f} {pct:>10.1f}%')

In [ ]:
# Visualize: smoothed states with missing data
smoothed_complete = res_complete.smoothed_state[:, 0]
smoothed_missing = res_missing.smoothed_state[:, 0]
se_missing = np.sqrt(res_missing.smoothed_cov[:, 0, 0])

fig, ax = plt.subplots(figsize=(14, 6))

# Observed data (with gaps)
obs_mask = ~np.isnan(y_missing)
ax.plot(years[obs_mask], y_missing[obs_mask], 'k.', markersize=5, alpha=0.5, label='Observed')
ax.plot(years[~obs_mask], y_nile[~obs_mask], 'rx', markersize=8, label='Missing (true value)')

# Smoothed states
ax.plot(years, smoothed_complete, 'b-', linewidth=1.2, alpha=0.7, label='Smoothed (complete data)')
ax.plot(years, smoothed_missing, 'r-', linewidth=1.2, label='Smoothed (10% missing)')
ax.fill_between(
    years,
    smoothed_missing - 1.96 * se_missing,
    smoothed_missing + 1.96 * se_missing,
    alpha=0.15, color='red', label='95% CI (missing data)'
)

ax.set_xlabel('Year')
ax.set_ylabel('Annual Flow')
ax.set_title('Kalman Filter Handles Missing Data Naturally')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

# Show that uncertainty increases at missing points
print('Smoothed state SE at missing vs observed points:')
print(f'  Mean SE at observed points:  {se_missing[obs_mask].mean():.2f}')
print(f'  Mean SE at missing points:   {se_missing[~obs_mask].mean():.2f}')
print(f'  Ratio: {se_missing[~obs_mask].mean() / se_missing[obs_mask].mean():.2f}x')

## Conclusions

### Key Takeaways

1. **ARIMA = State-Space**: Any ARIMA(p,d,q) model can be represented exactly as a linear
   Gaussian state-space model using the companion form. This includes seasonal models
   ARIMA(p,d,q)(P,D,Q)$_s$.

2. **ARIMA(0,1,1) $\equiv$ Local Level**: The simplest structural time series model (random
   walk + noise) is exactly equivalent to ARIMA(0,1,1) in its reduced form. The parameters
   map via the autocovariance structure.

3. **Estimation via Kalman Filter**: The Kalman filter prediction error decomposition provides
   exact maximum likelihood estimation, identical to direct MLE methods.

4. **Missing Data**: The Kalman filter handles missing observations naturally by skipping the
   update step. This is a major advantage over classical ARIMA estimation methods.

5. **Unified Framework**: The state-space representation provides a unified framework where
   ARIMA, structural models, and more complex specifications (time-varying parameters,
   multivariate models) all share the same estimation machinery.

### Advantages of the SSM Approach

| Feature | Classical ARIMA | State-Space ARIMA |
|---------|----------------|-------------------|
| Estimation | CSS or exact MLE | Kalman filter MLE |
| Missing data | Not supported | Natural |
| State extraction | No | Filtered + smoothed states |
| Forecasting | Recursive | State-based with uncertainty |
| Extensions | Limited | Unlimited (TVP, multivariate, ...) |